In [31]:
import kagglehub
import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.metrics import root_mean_squared_error
from itertools import combinations

## Load Data:

In [2]:
path = kagglehub.dataset_download("mirichoi0218/insurance")
print("Path to dataset files:", path)
files = os.listdir(path)
print("Files found in directory:", files)

Path to dataset files: /Users/johannzaroli/.cache/kagglehub/datasets/mirichoi0218/insurance/versions/1
Files found in directory: ['insurance.csv']


In [3]:
df = pd.read_csv(os.path.join(path, "insurance.csv"))
print(df.head())

# clean df, remove NaN values
df_cleaned = df.dropna()
len(df_cleaned)

   age     sex     bmi  children smoker     region      charges
0   19  female  27.900         0    yes  southwest  16884.92400
1   18    male  33.770         1     no  southeast   1725.55230
2   28    male  33.000         3     no  southeast   4449.46200
3   33    male  22.705         0     no  northwest  21984.47061
4   32    male  28.880         0     no  northwest   3866.85520


1338

In [4]:
# encode categorical columns
encoded_df = pd.get_dummies(df_cleaned, columns=['sex', 'smoker', 'region'], dtype=int)
encoded_df.head()

,age,bmi,children,charges,sex_female,sex_male,smoker_no,smoker_yes,region_northeast,region_northwest,region_southeast,region_southwest
0,19,27.900,0,16884.92400,1,0,0,1,0,0,0,1
1,18,33.770,1,1725.55230,0,1,1,0,0,0,1,0
2,28,33.000,3,4449.46200,0,1,1,0,0,0,1,0
3,33,22.705,0,21984.47061,0,1,1,0,0,1,0,0
4,32,28.880,0,3866.85520,0,1,1,0,0,1,0,0


## Feature engineering:

In [5]:
encoded_df["age_bmi_smoker_risk"] =  encoded_df["age"] * encoded_df["bmi"] * encoded_df["smoker_yes"].map({0: 1.0, 1: 2.5}) 
encoded_df["bmi_x_age"] = encoded_df["bmi"] * encoded_df["age"]
encoded_df.head()

,age,bmi,children,charges,sex_female,sex_male,smoker_no,smoker_yes,region_northeast,region_northwest,region_southeast,region_southwest,age_bmi_smoker_risk,bmi_x_age
0,19,27.900,0,16884.92400,1,0,0,1,0,0,0,1,1325.250,530.100
1,18,33.770,1,1725.55230,0,1,1,0,0,0,1,0,607.860,607.860
2,28,33.000,3,4449.46200,0,1,1,0,0,0,1,0,924.000,924.000
3,33,22.705,0,21984.47061,0,1,1,0,0,1,0,0,749.265,749.265
4,32,28.880,0,3866.85520,0,1,1,0,0,1,0,0,924.160,924.160


#### Observations:
Initially, I was going to create columns for bmi_x_smoker, age_x_smoker. However, these were essentially multiplying the smoker status (0 or 1) by the other feature and creating little to no signal for the model.   
  
After careful consideration, I created a unified risk score that is age_bmi_smoker_risk score where being a smoker scales the multiplied bmi*age values by another 2.5x, modeling the compounding effects of being a smoker, being of a certain age and having a higher bmi.

I also created a bmi_x_age feature to model the compounding effects of both. This may prove to be mostly redundant with the age_bmi_smoker_risk score.

Note: The transformed features are for the polynomial model, not the random forest.

## Scale Features:

In [6]:
# scale features down
encoded_df['sc_bmi'] = encoded_df['bmi']/10
encoded_df['sc_age'] = encoded_df['age']/10
encoded_df['sc_charges'] = encoded_df['charges']/1000
encoded_df['sc_age_bmi_smoker_risk'] = encoded_df['age_bmi_smoker_risk']/100
encoded_df['sc_bmi_x_age'] = encoded_df['bmi_x_age']/100

encoded_scaled_df = encoded_df[['sc_charges', 'sc_bmi', 'sc_age', 'sc_age_bmi_smoker_risk', 'sc_bmi_x_age', 'children',
                                'sex_female', 'sex_male', 'smoker_yes', 'smoker_no', 'region_northeast', 'region_northwest', 'region_southeast', 'region_southwest'
                               ]]
encoded_scaled_df.head()

,sc_charges,sc_bmi,sc_age,sc_age_bmi_smoker_risk,sc_bmi_x_age,children,sex_female,sex_male,smoker_yes,smoker_no,region_northeast,region_northwest,region_southeast,region_southwest
0,16.884924,2.7900,1.9,13.25250,5.30100,0,1,0,1,0,0,0,0,1
1,1.725552,3.3770,1.8,6.07860,6.07860,1,0,1,0,1,0,0,1,0
2,4.449462,3.3000,2.8,9.24000,9.24000,3,0,1,0,1,0,0,1,0
3,21.984471,2.2705,3.3,7.49265,7.49265,0,0,1,0,1,0,1,0,0
4,3.866855,2.8880,3.2,9.24160,9.24160,0,0,1,0,1,0,1,0,0


## Split Data:

In [7]:
X_train, X_test = train_test_split(encoded_scaled_df, test_size=0.2, random_state=42)

In [8]:
X_train.head()

,sc_charges,sc_bmi,sc_age,sc_age_bmi_smoker_risk,sc_bmi_x_age,children,sex_female,sex_male,smoker_yes,smoker_no,region_northeast,region_northwest,region_southeast,region_southwest
560,9.193839,1.995,4.6,9.1770,9.1770,2,1,0,0,1,0,1,0,0
1285,8.534672,2.432,4.7,11.4304,11.4304,0,1,0,0,1,1,0,0,0
1142,27.117994,2.486,5.2,12.9272,12.9272,0,1,0,0,1,0,0,1,0
969,8.596828,3.432,3.9,13.3848,13.3848,5,1,0,0,1,0,0,1,0
486,12.475351,2.147,5.4,11.5938,11.5938,3,1,0,0,1,0,1,0,0


## Polynomial Model:

### Training and Automated Sweep:

In [42]:
# Note: removed redundant features with perfect multicollinearity: sex_male, smoker_no, region_northeast
# Note: not included in final model: 'region_northwest', 'region_southeast', 'region_southwest', 'children', 'sex_female'
features = ['sc_bmi', 'sc_age', 'sc_age_bmi_smoker_risk', 'sc_bmi_x_age', 'smoker_yes']

max_r_squared = -np.inf
best_model_metrics = ''

for r in range(1, len(features) + 1):
    for combo in combinations(features, r):

        # build formula terms
        poly_terms = []
        for degree, feat in enumerate(combo, start=1):
            if feat == 'sex_female' or degree == 1:
                poly_terms.append(feat)
            else:
                poly_terms.append(f"np.power({feat}, {degree})")

        # fit model
        formula_string = 'sc_charges ~ ' + ' + '.join(poly_terms)
        res = smf.ols(formula=formula_string, data=X_train).fit()

        if res.rsquared > max_r_squared:
            max_r_squared = res.rsquared
            best_model_metrics = f"Formula: {formula_string}\nR-squared: {res.rsquared:.4f}\ncoefficients: {res.params}\n \n, t_values: \n {res.tvalues}\n p_values: \n {res.pvalues} "
        
print(best_model_metrics)

Formula: sc_charges ~ sc_bmi + np.power(sc_age, 2) + np.power(sc_age_bmi_smoker_risk, 3) + np.power(sc_bmi_x_age, 4) + np.power(smoker_yes, 5)
R-squared: 0.7650
coefficients: Intercept                              -7.080936
sc_bmi                                  3.425881
np.power(sc_age, 2)                     0.330883
np.power(sc_age_bmi_smoker_risk, 3)     0.000096
np.power(sc_bmi_x_age, 4)              -0.000018
np.power(smoker_yes, 5)                19.950365
dtype: float64
 
, t_values: 
 Intercept                              -5.611385
sc_bmi                                  9.065015
np.power(sc_age, 2)                    13.452310
np.power(sc_age_bmi_smoker_risk, 3)    10.135071
np.power(sc_bmi_x_age, 4)              -4.602698
np.power(smoker_yes, 5)                34.751844
dtype: float64
 p_values: 
 Intercept                               2.558873e-08
sc_bmi                                  5.848616e-19
np.power(sc_age, 2)                     3.267357e-38
np.power(sc_age_bmi

### Observations (with manual, backward elimination):

An initial model was trained with all features available (while omitting the redundant features). While this model had a slightly higher R^2 of 0.7663, the following features did not have significant p-values: region_northwest, region_southeast, region_southwest, children.  
  
Conducting backward elimination by training a new model without these features earned an R^2 of 0.7650. Its sex_female feature did not have a significant p-value and was dropped on the next iteration.  
  
The final model has highly significant features comprising of sc_bmi, sc_age, sc_age_bmi_smoker_risk, sc_bmi_x_age, smoker_yes with an R^2 of 0.7650. This model conserving the majority of the initial variance captured with the larger feature set.
  
Final formula: sc_charges ~ sc_bmi + np.power(sc_age, 2) + np.power(sc_age_bmi_smoker_risk, 3) + np.power(sc_bmi_x_age, 4) + np.power(smoker_yes, 5)

### RMSE Calculation

In [35]:
# Cross-Validation RMSE
formula = "sc_charges ~ sc_bmi + np.power(sc_age, 2) + np.power(sc_age_bmi_smoker_risk, 3) + np.power(sc_bmi_x_age, 4) + np.power(smoker_yes, 5)"
kf = KFold(n_splits=5, shuffle=True, random_state=42)
rmse_scores = []

for train_idx, val_idx in kf.split(X_train):
    # Fit on training split, predict on validation split
    model = smf.ols(formula, data=X_train.iloc[train_idx]).fit()
    preds = model.predict(X_train.iloc[val_idx])

    # Calculate and store RMSE
    actuals = X_train.iloc[val_idx]["sc_charges"]
    rmse_scores.append(root_mean_squared_error(actuals, preds))

print(f"Mean Cross-Validation RMSE: {np.mean(rmse_scores):.4f}")

Mean Cross-Validation RMSE: 5.8248


In [34]:
# Test RMSE
formula = "sc_charges ~ sc_bmi + np.power(sc_age, 2) + np.power(sc_age_bmi_smoker_risk, 3) + np.power(sc_bmi_x_age, 4) + np.power(smoker_yes, 5)"
final_model = smf.ols(formula, data=X_train).fit()
test_preds = final_model.predict(X_test)
test_actuals = X_test["sc_charges"]

test_rmse = root_mean_squared_error(test_actuals, test_preds)
print(f"Final Test RMSE: {test_rmse:.4f}")

Final Test RMSE: 5.5587


### Polynomial Model Conclusions:

The Mean Cross-Validation RMSE is 5.8248 and the Final Test RMSE is 5.5587. Since both of these number are close to one another, the model generalizes well to unseen data! The model did not overfit to the training data and learned the underlying pattern and variance of the data up to an R^2 of 0.7650. The pairplot helped the most in discovering a hidden relationship between charges and age, which was subsequentially modeled through the engineered features sc_age_bmi_smoker_risk and sc_bmi_x_age.

## Multi-Linear Model:
During EDA, it was established that the relationship between features was mostly non-linear and that a multi-linear model would not be as strong of a fit as a polynomial model. Out of curiosity, this model was created after the polynomial model was trained. It uses the best features from the polynomial model: sc_bmi, sc_age, sc_age_bmi_smoker_risk, sc_bmi_x_age, smoker_yes.

### Training:

In [47]:
formula = "sc_charges ~ sc_bmi + sc_age + sc_age_bmi_smoker_risk + sc_bmi_x_age + smoker_yes"
mod = smf.ols(formula=formula, data=X_train)
res = mod.fit()

res.rsquared
best_model_metrics = f"Formula: {formula}\nR-squared: {res.rsquared:.4f}\ncoefficients: {res.params}\n \n, t_values: \n {res.tvalues}\n p_values: \n {res.pvalues} "
        
print(best_model_metrics)

Formula: sc_charges ~ sc_bmi + sc_age + sc_age_bmi_smoker_risk + sc_bmi_x_age + smoker_yes
R-squared: 0.7647
coefficients: Intercept                -10.487239
sc_bmi                     3.602529
sc_age                     2.843440
sc_age_bmi_smoker_risk     0.613893
sc_bmi_x_age              -0.878631
smoker_yes                12.843754
dtype: float64
 
, t_values: 
 Intercept                 -3.904198
sc_bmi                     4.199342
sc_age                     4.273444
sc_age_bmi_smoker_risk    10.717405
sc_bmi_x_age              -3.974796
smoker_yes                11.645737
dtype: float64
 p_values: 
 Intercept                 1.004881e-04
sc_bmi                    2.900825e-05
sc_age                    2.097257e-05
sc_age_bmi_smoker_risk    1.602543e-25
sc_bmi_x_age              7.519212e-05
smoker_yes                1.379324e-29
dtype: float64 


### RMSE Calculation:

In [48]:
# Cross-Validation RMSE
formula = "sc_charges ~ sc_bmi + sc_age + sc_age_bmi_smoker_risk + sc_bmi_x_age + smoker_yes"
kf = KFold(n_splits=5, shuffle=True, random_state=42)
rmse_scores = []

for train_idx, val_idx in kf.split(X_train):
    # fit on training split, predict on validation split
    model = smf.ols(formula, data=X_train.iloc[train_idx]).fit()
    preds = model.predict(X_train.iloc[val_idx])

    # calculate and store RMSE
    actuals = X_train.iloc[val_idx]["sc_charges"]
    rmse_scores.append(root_mean_squared_error(actuals, preds))

print(f"Mean Cross-Validation RMSE: {np.mean(rmse_scores):.4f}")

Mean Cross-Validation RMSE: 5.8328


In [49]:
# Test RMSE
formula = "sc_charges ~ sc_bmi + sc_age + sc_age_bmi_smoker_risk + sc_bmi_x_age + smoker_yes"
final_model = smf.ols(formula, data=X_train).fit()
test_preds = final_model.predict(X_test)
test_actuals = X_test["sc_charges"]

test_rmse = root_mean_squared_error(test_actuals, test_preds)
print(f"Final Test RMSE: {test_rmse:.4f}")

Final Test RMSE: 5.5810


### Multi-Linear Model Conclusions:
The model performs nearly as well as the polynomial model. It has an R^2 of 0.7647, a Mean Cross-Validation RMSE of 5.8328 and a Final Test RMSE of 5.5810. Arguably, the features are doing most of the heavy lifting for predicting this amount of variance across the data. Let's see if a random forest improves performance!

In [ ]:
## Random Forest Model:
